In [0]:
%sql
CREATE Schema if NOT EXISTS pysparklearning.practice

In [0]:
%sql
CREATE VOLUME If NOT EXISTS pysparklearning.practice.raw_files

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window


In [0]:
raw_path="/Volumes/pysparklearning/practice/raw_files"

In [0]:
raw_df=(
    spark.read.option("header",True)
              .option("inferschema",false)
              .csv(raw_path)

)

In [0]:
raw_df1=(
    spark.read.option("header",True)
              .option("inferschema",false)
              .csv(raw_path)

)

In [0]:
string_cols=raw_df.columns
df=raw_df
for c in string_cols:
    df=df.withColumn(c,
                     f.when(f.trim(f.col(c))=="",None)
                     .otherwise(f.trim(f.col(c))))
    

In [0]:

df=df.withColumn("loan_type",f.upper(f.col("loan_type")))\
     .withColumn("loan_status",f.upper(f.col("loan_status")))
display(df)

In [0]:
display(df.describe)

In [0]:
df=raw_df

In [0]:
df=df.withColumn("loan_amount",f.expr("try_cast(loan_amount as decimal(18,1))"))\
     .withColumn("overdue_amount",f.expr("try_cast(overdue_amount as decimal(18,2))"))\
     .withColumn("overdue_date",f.expr("try_cast(overdue_date as date)"))\
     .withColumn("entry_timestamp",f.expr("try_cast(entry_timestamp as timestamp)"))        

In [0]:
df.printSchema()

In [0]:
valid_status=f.col("loan_status").isin("ACTIVE", "PENDING", "CLOSED")
display(valid_status)

In [0]:
df_filtered=df.filter(valid_status)
display(df_filtered)

In [0]:
invalid_status=~f.col("loan_status").isin("ACTIVE", "PENDING", "CLOSED")
df_invalid_status=df.filter(invalid_status)
display(df_invalid_status)

In [0]:
reason_expr=f.concat_ws("; ",
                        f.when(f.col("loan_no").isNull(),f.lit("missing_loan_no")),
                        f.when(f.col("loan_type").isNull(),f.lit("missing loan_type")),
                        f.when(f.col("loan_amount").isNull(),f.lit("loan_amount is missing")),
                        f.when(f.col("loan_amount")<0,f.lit("negative loan_amount")),
                        f.when(f.col("branch_code").isNull(), f.lit("missing_branch_code")),
                        f.when(f.col("overdue_date").isNull(), f.lit("missing_or_invalid_overdue_date")),
                        f.when(f.col("last_payment_date").isNull(), f.lit("missing_or_invalid_last_payment_date")),
                        f.when(f.col("entry_timestamp").isNull(), f.lit("missing_or_invalid_entry_timestamp")),
                        f.when(invalid_status, f.lit("invalid_loan_status"))
                        )


In [0]:
checked_df=df.withColumn("reject_reason",reason_expr)
valid_df=df.filter(f.col("reject_reason")=="")
rejected_df=df.filter(f.col("reject_reason")!="")
display(checked_df)

In [0]:
exact_duplicate_count=df.count()-df.dropDuplicates().count()
display(exact_duplicate_count)

In [0]:
duplicate_keys_df=(df.groupBy(col("loan_no"))
                     .count()
                                          
                   )

display(duplicate_keys_df)


In [0]:
duplicate_keys_df=(df.groupBy(col("loan_no"))
                     .count()
                     .filter(col("loan_no").isNotNull() & (col("count")>1))
                     
                   )
display(duplicate_keys_df)                   


In [0]:
window_spec=Window.partitionBy(col("loan_no")).orderBy(col("entry_timestamp").desc(),col("branch_code").asc())
latest_df=df.withColumn("rn",row_number().over(window_spec))\
            .filter(col("rn")==1)\
            .drop(col("rn"))
print(f"raw df count {df.count()}")   
print(f"df count after dedup {latest_df.count()}")        
# latest_df.count()

In [0]:
window_spec=Window.partitionBy(col("loan_no")).orderBy(col("entry_timestamp").desc(),col("branch_code").asc())
dup_df=df.withColumn("rn",row_number().over(window_spec))\
            .filter(col("rn")!=1)
            


In [0]:
display(dup_df)

In [0]:
enriched_df = (latest_df
    .withColumn("overdue_band",
        f.when(f.col("overdue_amount") == 0, "NO_OVERDUE")
         .when(f.col("overdue_amount") < 10000, "LOW")
         .when(f.col("overdue_amount") < 50000, "MEDIUM")
         .otherwise("HIGH"))
    .withColumn("overdue_ratio",
        f.when(f.col("loan_amount") > 0,
               f.round(f.col("overdue_amount") / f.col("loan_amount"), 4))
         .otherwise(None)))

In [0]:
enriched_df.createOrReplaceTempView("loan_cases_staged")

In [0]:
spark.sql(
    """
    select loan_type,sum(loan_amount) as totalAmount from
    loan_cases_staged
    group by loan_type
    order by totalAmount desc
   
    """
).show()

In [0]:
spark.sql(
    """
    select loan_type,count(distinct loan_no) as totalLoans from
    loan_cases_staged
    group by loan_type
    order by totalLoans desc
   
    """
).show()

In [0]:
spark.sql("""
          
          select branch_code,branch_name,sum(overdue_amount) as totalOverdue
          from loan_cases_staged
          group by 1,2
          
          """).show()

In [0]:
spark.sql("""
          
          select loan_status,count(loan_no) as totalLoans,sum(loan_amount) as totalLoanAmount
          from loan_cases_staged
          group by 1
          
          """).show()

In [0]:
spark.sql("""
          
          select branch_code,branch_name,sum(loan_amount) as totalLoanAmount
          from loan_cases_staged
          group by 1,2
          order by totalLoanAmount desc
          limit 3
          
          """).show()

In [0]:
spark.sql("""
          
         with branch_totals As (
              select branch_code,branch_name,sum(loan_amount) as totalLoanAmount
          from loan_cases_staged
          group by 1,2
          order by totalLoanAmount desc
          limit 3
         ),
         ranked_branches as (
             select *,dense_rank() over(partition by branch_code order by totalLoanAmount desc ) as rank1
             from branch_totals
         )
         select *
         from ranked_branches
         where rank1=1
          
          """).show()